# Ü5.1 — Erster ETL-Job: S3 → S3

Interaktive Glue-Session: `raw.orders` lesen → ApplyMapping (schmutzige
Spaltennamen bereinigen + Typen) → partitioniertes Parquet nach
`processed/orders/` schreiben und als `processed.orders` katalogisieren.

**Job vs. Notebook:** als Job kapselt `Job.init`/`job.commit()` den Lauf (und
trägt Bookmarks fort, siehe Ü8.1); interaktiv genügt der `getSink`-Write mit
Katalogisierung. Der Output entspricht dem des Ü5.1-Jobs.

**Voraussetzung:** Crawler aus Ü3.1 hat `raw.orders` katalogisiert.

**So arbeitest du:** Die Code-Zellen sind vorgefertigt und lauffähig — **ersetze
nur die `____`-Platzhalter**. Die Syntax steht schon; du entscheidest nur, WAS
wohin gehört. Über jeder Zelle steht zu jedem `____` eine kurze Leitfrage.

In [ ]:
%idle_timeout 60
%glue_version 5.0
%worker_type G.1X
%number_of_workers 2
# In Glue Studio wird die IAM-Rolle in der UI gewählt. Alternativ per Magic:
# %iam_role arn:aws:iam::<account>:role/AWSGlueServiceRole-GfuGlueTraining
%%configure
{ "--enable-glue-datacatalog": "true" }

In [ ]:
from awsglue.context import GlueContext
from awsglue.transforms import ApplyMapping
from pyspark.context import SparkContext

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

OUTPUT = "s3://gfu-glue-training-629452195361/processed/orders/"

## 1) Lesen — DynamicFrame aus dem Katalog

Die Zelle unten ist vorgefertigt — ersetze nur die `____`. Frage: Aus welcher
Katalog-Datenbank und welcher Tabelle kommen die Rohdaten (siehe Voraussetzung
oben)?

In [ ]:
# Fülle die ____ aus. Syntax steht schon — du entscheidest nur, WAS wohin gehört.
#   ____datenbank : aus welcher Katalog-Datenbank kommen die Rohdaten?
#   ____tabelle   : welche Tabelle darin enthält die Bestellungen?
orders = glueContext.create_dynamic_frame.from_catalog(
    database="____datenbank", table_name="____tabelle", transformation_ctx="orders"
)
orders.printSchema()

## 2) ApplyMapping — Spalten bereinigen + Typen setzen

Die Rohspalten haben Leerzeichen im Namen (z. B. `"cust id"`) und alles kommt als
`string` an. Ersetze nur die `____`: Wie heißt die Rohspalte, wie soll sie sauber
heißen, und welcher Datentyp passt zum Bestellbetrag? Die restlichen Zeilen sind
schon ausgefüllt — orientiere dich an ihnen.

In [ ]:
# Fülle die ____ aus. Syntax steht schon — du entscheidest nur, WAS wohin gehört.
# Eine Mapping-Zeile ist: ("<rohname>", "<rohtyp>", "<zielname>", "<zieltyp>").
#   ____quellspalte : wie heißt die Rohspalte im Katalog (mit Leerzeichen)?
#   ____zielspalte  : wie soll dieselbe Spalte danach sauber heißen?
#   ____zieltyp     : welcher Datentyp passt zum Bestellbetrag (nicht mehr string)?
mapped = ApplyMapping.apply(
    frame=orders,
    mappings=[
        ("____quellspalte", "string", "____zielspalte", "string"),
        ("order total", "string", "order_total", "____zieltyp"),
        ("order date", "string", "order_date", "string"),
        ("status", "string", "status", "string"),
    ],
    transformation_ctx="mapped",
)
mapped.printSchema()

## 3) Senke — partitioniertes Parquet + Katalog `processed.orders`

Die Zelle unten ist vorgefertigt — ersetze nur die `____`. Fragen: Nach welcher
Spalte partitionierst du sinnvoll? In welche Katalog-Datenbank und -Tabelle soll
das Ergebnis? Und welches Ausgabeformat schreibt Glue hier (Tipp: der
Glue-eigene Parquet-Writer heißt anders als nur `parquet`)?

In [ ]:
# Fülle die ____ aus. Syntax steht schon — du entscheidest nur, WAS wohin gehört.
#   ____partition     : nach welcher Spalte sollen die Parquet-Dateien partitioniert werden?
#   ____zieldatenbank : in welche Katalog-Datenbank kommt die Ergebnistabelle?
#   ____zieltabelle   : wie soll die Ergebnistabelle heißen?
#   ____format        : welchen Format-String erwartet Glue für sein Parquet?
sink = glueContext.getSink(
    path=OUTPUT,
    connection_type="s3",
    updateBehavior="UPDATE_IN_DATABASE",
    partitionKeys=["____partition"],
    enableUpdateCatalog=True,
    transformation_ctx="sink_orders",
)
sink.setCatalogInfo(catalogDatabase="____zieldatenbank", catalogTableName="____zieltabelle")
sink.setFormat("____format")
sink.writeFrame(mapped)
print("geschrieben + katalogisiert ->", OUTPUT)